#Time Series - LAB 1

In [ ]:
!pip install hampel #install library

#import libraries
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from hampel import hampel
import warnings
warnings.filterwarnings('ignore')

##Data Visualization

In [ ]:
series = pd.read_csv('/content/daily-min-temperatures.txt',index_col=0, header=0, parse_dates=True) #read the csv file with the data
#header = 0: contain information about column names
#parse_date = True, inform that the dataset has a date column
#index_col = 0: column that will be the index of the time series

#This dataset describes the minimum daily temperatures over 10 years (1981-1990) in the city Melbourne, Australia.
#The units are in degrees Celsius and there are 3,650 observations. The source of the data is credited as the Australian Bureau of Meteorology.

series.head() #first values from the time series


In [ ]:
series.plot() #plot time series
plt.show()
series.plot(style='k.') #plot time series using dots (.) in black (k)
plt.show()
plt.stem(series.Temp[1:50]) #plot time series using stems

In [ ]:
series.hist() #plot histogram
plt.show()

series.plot(kind='kde') #plot using kernel density estimate: It is a non-parametric way to estimate the probability density function (PDF) of a random variable.
plt.show()

In [ ]:
groups = series.groupby(pd.Grouper(freq='YE')) #Group data by year. YE: years
years = pd.DataFrame()
for name, group in groups:
  years[name.year] = group.Temp.values
years.plot(subplots=True, legend=False)
plt.show()

display(years)

In [ ]:
sns.boxplot(data=years) #show the boxplot of each year
plt.show()

In [ ]:
sns.heatmap(years, cbar_kws={"label": "Temperature"}) #show the heatmap
plt.title("Daily Minimum Temperatures")
plt.xlabel("Year")
plt.ylabel("Day")
plt.show()

In [ ]:
#A useful type of plot to explore the relationship between each observation and a lag of that observation is called the scatter plot.
lag = 1
pd.plotting.lag_plot(series,lag)
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
plot_acf(series, lags=50) #plot autocorrelation function (ACF)
plt.show()
plot_pacf(series, lags=50) #plot partial autocorrelation function (PACF)
plt.show()

#Data Manipulation and Statistical Descriptors

In [10]:
serie = pd.Series([10, 20, 77, 6, -150, 70, 86, 93, 123]) #create an one-dimensional array.
serie.name = 'Série'

In [ ]:
print(f'Dimensions: {serie.shape}') #return a tuple representing the dimensionality of the data
print(f'First rows:\n{serie.head(n = 5)}') #return the first n rows
print(f'Last rows:\n{serie.tail(n = 5)}') #return the last n rows
print(f'Value at index 1: {serie[1]}') #return value at index 1
print(f'Values from index 4 to last index:\n{serie[4:]}') #return values from index 4 to last index
print(f'Values from index 4 to index 6:\n{serie[4:7]}') #return values from index 4 to index 7 (7 is not included)
print(f'Values from index 0 to index 4:\n{serie[:5]}') #return values from index 0 to index 5 (5 is not included)

In [ ]:
display(serie)
serie[1] = -100 #change the value at index 1
print(serie[1])
serie[1:4] = [0, 0, 0] #change the values at indexes 1 to 3
print(serie[1:4])
serie[[1, 4, 7]] = [-200, -200, -200] #change the values at indexes 1, 4 and 7
print(serie[[1, 4, 7]])

In [ ]:
data = pd.read_csv('/content/daily-min-temperatures.txt',index_col=0, header=0, parse_dates=True)

indx = data.index
data = [data.iloc[i, 0] for i in range(len(indx))]

time_series = pd.Series(data, index = pd.date_range("1-1-1981", periods=len(data), freq="D"),  name="Daily Minimum Temperatures") #One-dimensional array.
time_series.plot() #line plot
plt.show()
time_series.plot.box() #box plot
plt.show()
time_series.plot.hist() #histogram
plt.show()

time_series.describe() #some statistics of the dataset

In [ ]:
data = pd.read_csv('/content/daily-min-temperatures.txt',index_col=0, header=0, parse_dates=True)
data.describe() #some statistics of the dataset


In [ ]:
print(time_series.std()) #standard deviation
print(time_series.var()) #variance
print(time_series.mean()) #mean
print(time_series.min()) #minimum
print(time_series.max()) #maximum
print(time_series.median()) #median
print(time_series.skew()) #skewness
print(time_series.kurtosis()) #kurtosis
print(time_series.max() - time_series.min()) #interval

In [ ]:
#Multivariate Time Series
columns = 5
rows = 100
matrix = np.random.normal(loc = 100, scale = 30, size = (rows, columns)).round(2)
time = pd.date_range(start = '2021-1-1', periods = rows, freq = 'D')
df = pd.DataFrame(matrix, columns = list('ABCDE'), index = time)
df.head()
df.describe()

In [ ]:
print(df['A'].mean())
print(df['E'].std())

In [ ]:
df.insert(loc = 5, column = 'Mean', value = df.mean(axis = 1)) #add a column
df.head()

In [ ]:
df.drop(labels = 'Mean', axis = 1) #delete a column

In [ ]:
display(df.loc['2021-1-1':'2021-1-10', 'A']) #Select one period and a column
display(df.iloc[:15, 2:4]) #Select one period and two columns

In [ ]:
df.iloc[:,1:3].plot() #line plot
plt.show()
df.plot.box() #box plot
plt.show()
df['B'].plot.hist() #histogram
plt.show()

##Treatment of Missing Data and Outliers

In [ ]:
series = pd.read_csv('/content/daily-min-temperatures.txt',index_col=0, header=0, parse_dates=True)

index = np.random.permutation(len(series))
series.Temp[index[:int(len(series)*0.05)]] = np.NaN #add 5% of missing data in the dataset
#series['Temp'][index[:int(len(series)*0.05)]] = np.NaN

series.plot()
plt.show()

nul_data = pd.isnull(series['Temp'])
series[nul_data]

original_series = series.copy() #copy of data

In [ ]:
# fill the missing data using the mean of the present observations
series = original_series.copy()
series = series.Temp.fillna(series.Temp.mean()) #Replace missing values with the mean
#series = series.Temp.fillna(series.Temp.median()) #Replace missing values with the median
#series = series.fillna(method ='bfill') #Last Observation Carried Forward (LOCF): Replaces missing values with the last known value
#series = series.fillna(method ='ffill') #Next Observation Carried Backward (NOCB): Replaces missing values with the next known value.
series[nul_data]

In [ ]:
series = original_series.copy()

series['Temp'] = series['Temp'].interpolate(method='linear') #Linear Interpolation: Estimates missing values by drawing a straight line between the two nearest known data points.
#series['Temp'] = series['Temp'].interpolate(option='spline') #Spline Interpolation: Estimates missing values by fitting a flexible, curved line through the data points.
series[nul_data]


In [ ]:
original_series = pd.read_csv('/content/daily-min-temperatures.txt',index_col=0, header=0, parse_dates=True)

index = np.random.permutation(len(series))
original_series.Temp[index[:int(len(series)*0.01)]] = 3*original_series.Temp[index[:int(len(original_series)*0.01)]] #add 1% of outliers in the dataset

original_series.plot()
plt.show()

series = original_series.copy()

#Hampel function detects and treats outliers
#The hampel function has three available parameters:
#data: The input 1-dimensional data to be filtered (pandas.Series or numpy.ndarray).
#window_size (optional): The size of the moving window for outlier detection (default is 5).
#n_sigma (optional): The number of standard deviations for outlier detection (default is 3.0).
result = hampel(series['Temp'], window_size=11)
series['Temp'][:] = result.filtered_data
series.plot()
plt.show()

outlier_indices = result.outlier_indices
print(f'Outliers that were detected:\n{outlier_indices}') #outliers that were detected
for index in index[:int(len(series)*0.01)]:
  if index not in outlier_indices:
    print(False) #Added outliers that were not detected


In [ ]:
filtered_data = result.filtered_data
outlier_indices = result.outlier_indices
medians = result.medians
mad_values = result.median_absolute_deviations
thresholds = result.thresholds

for i in range(5):
  medians[i] = 0
  thresholds[i] = 0
  medians[-i] = 0
  thresholds[-i] = 0

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Plot the original data with estimated standard deviations in the first subplot
axes[0].plot(filtered_data, label='Original Data', color='b')
axes[0].fill_between(range(len(filtered_data)), medians + thresholds, medians - thresholds, color='gray', alpha=0.5, label='Median +- Threshold')
axes[0].set_xlabel('Data Point')
axes[0].set_ylabel('Value')
axes[0].set_title('Original Data with Bands representing Upper and Lower limits')

for i in outlier_indices:
    axes[0].plot(i, original_series['Temp'][i], 'ro', markersize=5)  # Mark as red

axes[0].legend()

# Plot the filtered data in the second subplot
axes[1].plot(filtered_data, label='Filtered Data', color='g')
axes[1].set_xlabel('Data Point')
axes[1].set_ylabel('Value')
axes[1].set_title('Filtered Data')
axes[1].legend()

# Adjust spacing between subplots
plt.tight_layout()

# Show the plots
plt.show()

##Frequency Domain

In [ ]:
# sampling rate
fs = 2000
# sampling interval
ts = 1.0/fs
t = np.arange(0,1,ts)

freq1 = 10
freq2 = 50
x = 3*np.sin(2*np.pi*freq1*t) + 0.5*np.sin(2*np.pi*freq2*t) + 0.5*np.random.normal(0,1,len(t)) #signal

plt.figure(figsize = (8, 6))
plt.plot(t, x, 'r')
plt.ylabel('Amplitude')

plt.show()

In [ ]:
X = np.fft.fft(x)/len(x)
X = np.abs(X[range(int(len(x)/2))])
X[1:] = 2*X[1:]

N = int(len(x)/2)
n = np.arange(N)
T = N/(fs/2)
freq = n/T

plt.figure(figsize = (12, 6))
plt.subplot(121)

plt.stem(freq, X, 'b', markerfmt=" ", basefmt="-b")
#plt.plot(freq, X, 'b')
plt.xlabel('Freq (Hz)')
plt.ylabel('FFT Amplitude |X(freq)|')
plt.xlim(0, 200)

plt.subplot(122)
plt.plot(t, x, 'r')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

In [ ]:
from scipy import signal

#Apply Butterworth Filter
fc = 30  # Cut-off frequency of the filter
fcn = fc / (fs / 2) # Normalize the frequency
order = 5
b, a = signal.butter(order, fcn, 'low')
#b, a = signal.butter(order, fcn, 'high')
output = signal.filtfilt(b, a, x)


OUT = np.fft.fft(output)/len(output)
OUT = np.abs(OUT[range(int(len(output)/2))])
OUT[1:] = 2*OUT[1:]

N = int(len(x)/2)
n = np.arange(N)
T = N/(fs/2)
freq = n/T

plt.figure(figsize = (12, 6))
plt.subplot(121)
plt.stem(freq, OUT, 'b', markerfmt=" ", basefmt="-b")
#plt.plot(freq, OUT, 'b')
plt.xlabel('Freq (Hz)')
plt.ylabel('FFT Amplitude |OUT(freq)|')
plt.xlim(0, 200)

plt.subplot(122)
plt.plot(t, output, 'r', label='filtered')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.legend()
plt.show()